# 02 — ML-Powered DQ Profiler

**Project:** dq-ml-impact-lab  
**Author:** Salini Anbalagan  
**Dataset:** UCI Bank Marketing Dataset  

---

## Notebook Objectives

1. Run the `DQProfiler` on the clean baseline dataset
2. Inspect per-column DQ scores across three dimensions:
   - **Completeness** — proportion of non-null values
   - **Consistency** — distributional stability (IQR-based / entropy-based)
   - **Anomaly Rate** — Isolation Forest anomaly detection (numeric only)
3. Generate and interpret the **DQ Scorecard**
4. Identify columns at risk before any degradation is applied
5. Save the baseline scorecard for comparison in Notebook 04

---

> **Key question for this notebook:**  
> *Which columns are already at risk in the clean dataset, and why?*

## 0. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.utils import load_dataset, save_scorecard, grade_color, format_score, print_scorecard
from src.dq_profiler import DQProfiler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13

print('Setup complete.')

## 1. Load Baseline Dataset

In [ ]:
df = load_dataset('../data/raw/bank-additional-full.csv', sep=';', target_col='y')
print(f'Dataset loaded: {df.shape}')

## 2. Run DQProfiler

In [ ]:
profiler = DQProfiler(df, random_seed=42, contamination=0.05)
print(profiler)

In [ ]:
# Run full scorecard
scorecard = profiler.score()
print(f'\nScorecard computed for {len(scorecard)} columns.')
print(f'Dataset DQ Score: {profiler.dataset_dq_score}')

## 3. Inspect the DQ Scorecard

In [ ]:
# Full scorecard — styled
scorecard.style.background_gradient(
    subset=['completeness', 'consistency', 'overall_dq'], cmap='RdYlGn'
).background_gradient(
    subset=['anomaly_rate'], cmap='RdYlGn'
).format({
    'completeness': '{:.4f}',
    'consistency': '{:.4f}',
    'anomaly_rate': lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A',
    'overall_dq': '{:.4f}',
})

In [ ]:
# Pretty print worst columns
print_scorecard(scorecard, top_n=5)

In [ ]:
# Dataset-level summary
summary = profiler.summary()
print('\nDataset-Level DQ Summary:')
for k, v in summary.items():
    print(f'  {k:<25}: {v}')

## 4. DQ Scorecard Visualisation

In [ ]:
# Overall DQ score per column — horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 7))

colors = [grade_color(g) for g in scorecard['dq_grade']]
bars = ax.barh(scorecard['column'], scorecard['overall_dq'], color=colors, edgecolor='white', height=0.7)

# Reference lines
ax.axvline(0.85, color='#2ecc71', linestyle='--', linewidth=1.2, label='A threshold (0.85)')
ax.axvline(0.70, color='#a8e063', linestyle='--', linewidth=1.2, label='B threshold (0.70)')
ax.axvline(0.55, color='#f39c12', linestyle='--', linewidth=1.2, label='C threshold (0.55)')
ax.axvline(0.40, color='#e74c3c', linestyle='--', linewidth=1.2, label='D threshold (0.40)')

# Value labels
for bar, score in zip(bars, scorecard['overall_dq']):
    ax.text(score + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{score:.3f}', va='center', fontsize=8.5)

ax.set_xlabel('Overall DQ Score')
ax.set_title('DQ Scorecard — Overall Score per Column (Baseline)', fontsize=14)
ax.set_xlim(0, 1.08)
ax.invert_yaxis()
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('../data/degraded/dq_scorecard_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dimension breakdown — grouped bar chart
dim_cols = ['completeness', 'consistency']
plot_df = scorecard.set_index('column')[dim_cols].copy()

# Add anomaly_rate where available
if 'anomaly_rate' in scorecard.columns:
    plot_df['anomaly_rate'] = scorecard.set_index('column')['anomaly_rate']

plot_df.sort_values('completeness', inplace=True)

ax = plot_df.plot(
    kind='barh',
    figsize=(14, 8),
    color=['#3498db', '#e67e22', '#2ecc71'],
    edgecolor='white',
    width=0.7
)
ax.set_xlabel('Score')
ax.set_title('DQ Dimension Scores per Column — Baseline', fontsize=14)
ax.set_xlim(0, 1.05)
ax.legend(['Completeness', 'Consistency', 'Anomaly Rate'], loc='lower right')
plt.tight_layout()
plt.savefig('../data/degraded/dq_dimensions_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. At-Risk Columns

In [ ]:
at_risk = profiler.at_risk_columns(threshold=0.70)

if at_risk.empty:
    print('No columns below the 0.70 DQ threshold in the baseline dataset.')
else:
    print(f'{len(at_risk)} column(s) at risk (overall_dq < 0.70):')
    display(at_risk)

In [ ]:
# Grade distribution
grade_dist = scorecard['dq_grade'].value_counts().sort_index()
grade_colors = [grade_color(g) for g in grade_dist.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(grade_dist.index, grade_dist.values, color=grade_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, grade_dist.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha='center', fontsize=11, fontweight='bold')
ax.set_title('DQ Grade Distribution — Baseline', fontsize=13)
ax.set_xlabel('Grade')
ax.set_ylabel('Number of Columns')
plt.tight_layout()
plt.savefig('../data/degraded/dq_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Baseline Scorecard

In [ ]:
save_path = save_scorecard(scorecard, '../data/degraded/baseline_scorecard.csv')
print(f'Baseline scorecard saved to: {save_path}')

## 7. Summary of Findings

In [ ]:
print('=' * 60)
print('  DQ PROFILER — KEY FINDINGS SUMMARY')
print('=' * 60)
print(f"""
Dataset DQ Score    : {profiler.dataset_dq_score}
Columns Profiled    : {len(scorecard)}
Columns At Risk     : {summary['n_columns_at_risk']} (overall_dq < 0.60)
Worst Column        : {summary['worst_column']}
Best Column         : {summary['best_column']}
Mean Completeness   : {summary['mean_completeness']}
Mean Consistency    : {summary['mean_consistency']}
Mean Anomaly Rate   : {summary['mean_anomaly_rate']}
""")
print('Interpretation:')
print('  - Completeness is generally high (expected for structured survey data)')
print('  - Consistency varies — pdays and campaign show distributional skew')
print('  - Anomaly rate captures extreme outliers in numeric features')
print('  - Baseline scorecard saved as reference for post-degradation comparison')
print('=' * 60)

---

## Next Steps

→ **Notebook 03:** `03_degradation_experiments.ipynb` — Inject controlled DQ issues  
→ **Notebook 04:** `04_ml_impact_analysis.ipynb` — Measure ML performance drop vs DQ score

---
*dq-ml-impact-lab | Salini Anbalagan*